| Step | Purpose |
|---|---|
| 4.0 | Install geopandas; extend the state list in `rx/places.py` |
| 4.1 | Load Newberry counties; keep versions 1763–1870; drop non-county areas |
| 4.2 | One row per county ID; a map point from the version in force by 1850 |
| 4.3 | Match key (`county_key`), region, first/last dates |
| 4.4 | Add one row per state (the fallback place) + save `processed/places` |

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

In [2]:
import pandas as pd, geopandas as gpd
SHP = config.RAW_DIR / "gis/newberry-ahcb-counties/shp/US_HistCounties_Gen001.shp"
c = gpd.read_file(SHP)
c = c[~c.NAME.str.contains(r"\bNCA\b|NON-COUNTY", case=False)]
c = c[(c.START_DATE <= "1870-12-31") & (c.END_DATE >= "1763-01-01")].copy()
print(len(c), c.ID.nunique(), c.NAME.nunique())


10921 2649 1754


In [3]:
base = c.STATE_TERR.str.replace(r"\s+(Colony|Territory|Republic)$", "", regex=True)
c["state"] = places.normalise_state_series(base.str.lower()).fillna(c.STATE_TERR)

c = c.sort_values(["ID", "START_DATE"])
anchor = c[c.START_DATE <= "1850-12-31"].groupby("ID").tail(1).to_crs("ESRI:102003")
pt = anchor.representative_point().to_crs("EPSG:4326")
pl = pd.DataFrame({"place_id": anchor.ID.values, "county": anchor.NAME.str.title().values,
                   "state": anchor.state.values, "lon": pt.x.values.round(4), "lat": pt.y.values.round(4)})
span = c.groupby("ID").agg(first_date=("START_DATE", "min"), last_date=("END_DATE", "max"))
pl = pl.join(span, on="place_id")
pl["county_key"] = (pl.county.str.lower().str.replace(r"^saint\s+", "st ", regex=True)
                      .str.replace(r"[^a-z ]", "", regex=True).str.strip())
WEST = {"KY", "TN", "OH", "IN", "IL", "MO", "AL", "MS", "MI", "LA", "AR"}
pl["region"] = pl.state.map(lambda s: "west" if s in WEST else "east")
pl = pl.drop_duplicates(["state", "county_key"])
print(len(pl), pl.state.nunique(), pl.county.nunique())

1875 44 1166


In [4]:
st = (pl.groupby("state").agg(lon=("lon", "mean"), lat=("lat", "mean"),
                              first_date=("first_date", "min"), last_date=("last_date", "max")).reset_index())
st["place_id"] = st.state; st["county"] = pd.NA; st["county_key"] = pd.NA
st["region"] = st.state.map(lambda s: "west" if s in WEST else "east")
pl = pd.concat([pl, st[pl.columns]], ignore_index=True)

assert pl.place_id.is_unique
io.write_stage(pl, "processed", "places")
pl.shape

(1919, 9)